# Hierarchical emotion model

Recovered from the supplied `test7.ipynb`. This public notebook contains the
primary training/evaluation flow and selected calibration/stability analysis,
with stored outputs and metadata removed. Run in a fresh kernel.

First install the separate training dependencies and run
`python scripts/prepare_experiment.py Dataset` from the repository root.
This creates local `outputs_hier_strategy4` split files.

These cells have been checked for Python syntax. Preprocessing was executed;
RoBERTa training and inference have **not** been rerun here. Trained weights and
threshold files are not bundled. The reported thesis scores are not outputs of
this cleaned notebook release. See `docs/NOTEBOOKS.md` for provenance and limits.


In [ ]:
from pathlib import Path
import os
# Start Jupyter from the repository root; this also supports the default
# notebook-directory working directory without personal absolute paths.
if Path.cwd().name == "notebooks" and (Path.cwd().parent / "README.md").is_file():
    os.chdir(Path.cwd().parent)
print("Working directory: repository root" if (Path.cwd()/"README.md").is_file() else "Check working directory")


## Source training notebook cell 13


In [ ]:
import os, re, ast, html, unicodedata, json, math, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

def seed_everything(seed=123):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(123)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

## Source training notebook cell 15


In [ ]:
import os
import json
import pandas as pd

# Use cfg.out_dir if CFG is already created; otherwise default to the canonical folder.
DATA_DIR = getattr(globals().get("cfg", None), "out_dir", "outputs_hier_strategy4")

TRAIN_PATH = os.path.join(DATA_DIR, "train.jsonl")
VAL_PATH = os.path.join(DATA_DIR, "val.jsonl")
TEST_PATH = os.path.join(DATA_DIR, "test.jsonl")
LABELS_PATH = os.path.join(DATA_DIR, "labels.json")

assert os.path.exists(TRAIN_PATH), f"Missing file: {TRAIN_PATH}"
assert os.path.exists(VAL_PATH), f"Missing file: {VAL_PATH}"
assert os.path.exists(TEST_PATH), f"Missing file: {TEST_PATH}"
assert os.path.exists(LABELS_PATH), f"Missing file: {LABELS_PATH}"

train_df = pd.read_json(TRAIN_PATH, lines=True)
val_df = pd.read_json(VAL_PATH, lines=True)
test_df = pd.read_json(TEST_PATH, lines=True)

with open(LABELS_PATH, "r", encoding="utf-8") as f:
    LABELS = json.load(f)

print("Loaded labels:", LABELS)
print(f"Loaded splits from: {DATA_DIR}")
print("  Train:", len(train_df))
print("  Val:  ", len(val_df))
print("  Test: ", len(test_df))

display(train_df.head(2))


## Source training notebook cell 16


In [ ]:


from dataclasses import dataclass

@dataclass
class CFG:
    model_name: str = "roberta-base"

    max_sents: int = 8     
    max_sent_len: int = 64    
    tail_frac: float = 0.30
    agg_layers: int = 2
    agg_heads: int = 8
    dropout: float = 0.1

    batch_size: int = 4
    accum_steps: int = 4
    epochs: int = 20          
    lr: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    grad_clip: float = 1.0

    device: str = DEVICE
    out_dir: str = "outputs_hier_strategy4"


cfg = CFG()
os.makedirs(cfg.out_dir, exist_ok=True)
print(cfg)


## Source training notebook cell 17


In [ ]:

import re
import numpy as np
from textblob import TextBlob

NEG_WORDS = set(["not", "no", "never", "n't", "cannot", "can't", "dont", "don't", "didnt", "didn't", "wont", "won't"])
THIRD_PRON = set(["he", "she", "they", "his", "her", "their", "him", "them"])
QUOTE_RE = re.compile(r"['\"“”‘’]")


def sent_split(text: str):
    if not text:
        return []
    text = text.replace("\n", " ")
    sents = re.split(r"(?<=[\.\!\?])\s+", text)
    sents = [s.strip() for s in sents if s.strip()]
    return sents


BD_TOKENS = {
    "dissociate","dissociation","dissociated",
    "foggy","fatigue","exhausted","apathetic",
    "adhd","burnout"
}
BD_PHRASES = {
    "brain fog", "spacing out",
    "can't focus", "cant focus", "cannot focus",
    "can't function", "cant function", "cannot function",
    "can't get out of bed", "cant get out of bed", "cannot get out of bed",
    "i haven't showered", "haven't showered", "cant shower", "can't shower",
    "can't bring myself", "cant bring myself", "cannot bring myself",
}

MEM_TOKENS = {"forget","forgot","forgetting","forgotten","memory","remember","remembering"}

def context_feats(sent: str):
    if not sent:
        return np.zeros(7, dtype=np.float32) 

    low = sent.lower()
    toks = re.findall(r"[a-z']+", low)

  
    neg   = 1.0 if any(t in NEG_WORDS for t in toks) else 0.0
    quote = 1.0 if QUOTE_RE.search(sent) else 0.0
    third = 1.0 if any(t in THIRD_PRON for t in toks) else 0.0
    bd    = 1.0 if any(t in BD_TOKENS for t in toks) else 0.0
    mem   = 1.0 if any(t in MEM_TOKENS for t in toks) else 0.0


    blob = TextBlob(sent)
    polarity = (blob.sentiment.polarity + 1) / 2 
    subjectivity = blob.sentiment.subjectivity

    return np.array([neg, quote, third, bd, mem, polarity, subjectivity], dtype=np.float32)

print("✅ context_feats function updated to return 7 dimensions.")

## Source training notebook cell 18


In [ ]:

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, use_fast=True)


class HierDataset(Dataset):
    def __init__(self, df_: pd.DataFrame):
        self.texts = df_["text_clean"].astype(str).tolist()
        self.ys = df_["y"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        y = torch.tensor(self.ys[idx], dtype=torch.float32)

        sents = sent_split(text)
        if len(sents) == 0:
            sents = [""]

        if len(sents) > cfg.max_sents:
            idx = np.linspace(0, len(sents) - 1, cfg.max_sents)
            idx = np.unique(np.round(idx).astype(int))
            sents = [sents[i] for i in idx]
        else:
            sents = sents[:cfg.max_sents]


        feats = np.stack([context_feats(s) for s in sents], axis=0)  
        return {"sents": sents, "feats": feats, "y": y}


def collate_fn(batch):
    B = len(batch)
    maxS = max(len(x["sents"]) for x in batch)
    maxS = min(maxS, cfg.max_sents)

    sents_2d, feats_3d = [], []
    y = torch.stack([x["y"] for x in batch], dim=0)

    for x in batch:
        sents = x["sents"][:maxS]
        feats = x["feats"][:maxS] 
        D = feats.shape[1] 

        pad_n = maxS - len(sents)
        if pad_n > 0:
            sents = sents + [""] * pad_n
            padding = np.zeros((pad_n, D), dtype=np.float32) 
            feats = np.concatenate([feats, padding], axis=0)


        sents_2d.append(sents)
        feats_3d.append(feats)

    feats_3d = torch.tensor(np.stack(feats_3d, axis=0), dtype=torch.float32)  
    flat_sents = [s for row in sents_2d for s in row]

    enc = tokenizer(
        flat_sents,
        padding=True,
        truncation=True,
        max_length=cfg.max_sent_len,
        return_tensors="pt",
    )

    sent_mask = torch.tensor([[1 if s.strip() else 0 for s in row] for row in sents_2d], dtype=torch.bool)

    return {
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "feats": feats_3d,
        "sent_mask": sent_mask,
        "y": y,
        "raw_sents": sents_2d
    }

pin = cfg.device.startswith("cuda")  
train_loader = DataLoader(
    HierDataset(train_df),
    batch_size=cfg.batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=pin
)
val_loader = DataLoader(
    HierDataset(val_df),
    batch_size=cfg.batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=pin
)
test_loader = DataLoader(
    HierDataset(test_df),
    batch_size=cfg.batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=pin
)

print("Loaders ready.")


## Source training notebook cell 19


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel
import math


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=256):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
  
        x = x + self.pe[:, :x.size(1), :]
        return x

class HierLabelWiseModel(nn.Module):
    def __init__(self, model_name, n_labels):
        super().__init__()
        self.n_labels = n_labels
        
  
        self.sent_encoder = AutoModel.from_pretrained(model_name)
        hidden = self.sent_encoder.config.hidden_size

        
        self.gate_net = nn.Sequential(
            nn.Linear(7, 16),        
            nn.LayerNorm(16),        
            nn.ReLU(),               
            nn.Dropout(0.5),         
            nn.Linear(16, 1),        
            nn.Sigmoid()             
        )

        
        self.rule_alpha = nn.Parameter(torch.tensor(0.5))

        enc_layer = nn.TransformerEncoderLayer(
            d_model=hidden, 
            nhead=cfg.agg_heads, 
            dim_feedforward=hidden*4, 
            dropout=cfg.dropout, 
            batch_first=True, 
            activation="gelu", 
            norm_first=True
        )
        self.agg = nn.TransformerEncoder(enc_layer, num_layers=cfg.agg_layers)
        self.pos = PositionalEncoding(hidden, max_len=cfg.max_sents)
        self.dropout = nn.Dropout(cfg.dropout)

        self.label_q = nn.Parameter(torch.randn(n_labels, hidden) * 0.02)
        self.cls_w = nn.Parameter(torch.randn(n_labels, hidden) * 0.02)
        self.cls_b = nn.Parameter(torch.zeros(n_labels))

    def forward(self, input_ids, attention_mask, feats, sent_mask):
        B, S, _ = feats.shape
        
        out = self.sent_encoder(input_ids=input_ids, attention_mask=attention_mask)
        sent_emb = out.last_hidden_state[:, 0, :].view(B, S, -1)

        gate = self.gate_net(feats)          
        sent_emb = sent_emb * (1.0 + gate)   

        sent_emb = sent_emb.masked_fill(~sent_mask.unsqueeze(-1), 0.0)

        x = self.pos(sent_emb)
        x = self.dropout(x)
        x = self.agg(x, src_key_padding_mask=~sent_mask)

        with torch.amp.autocast('cuda', enabled=False):
            x_fp = x.float()
            scores = torch.einsum("lh,bsh->bls", self.label_q.float(), x_fp)
            
            if feats.shape[-1] >= 5:
                prior = (feats[:, :, 3] + feats[:, :, 4]).float() * sent_mask.float()
                
             
                FORGET_IDX = 1 
                scores[:, FORGET_IDX, :] = scores[:, FORGET_IDX, :] + (self.rule_alpha * prior)

            scores = scores.masked_fill(~sent_mask.unsqueeze(1), -1e4)
            attn = torch.softmax(scores, dim=-1)
            
            label_rep = torch.einsum("bls,bsh->blh", attn, x_fp)
            logits = torch.einsum("blh,lh->bl", label_rep, self.cls_w.float()) + self.cls_b.float()
        
        return logits, attn

print("✅ 模型定义已更新 (MLP Gating + Learnable Alpha)")

## Source training notebook cell 20


In [ ]:
import torch.nn.functional as F

pos_weight = None 

class AsymmetricLoss(torch.nn.Module):
    def __init__(self, pos_weight=None, gamma_pos=0.0, gamma_neg=4.0):
        super().__init__()
        self.pos_weight = pos_weight
        self.gp = gamma_pos
        self.gn = gamma_neg

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(
            logits, targets, reduction="none", pos_weight=self.pos_weight
        )
        prob = torch.sigmoid(logits)
        p_t = prob * targets + (1 - prob) * (1 - targets)
        

        gamma = self.gp * targets + self.gn * (1 - targets)
        mod = (1 - p_t).clamp_min(1e-6).pow(gamma)
        
        return (bce * mod).mean()

criterion = AsymmetricLoss(pos_weight=None, gamma_pos=0.0, gamma_neg=4.0)

model = HierLabelWiseModel(cfg.model_name, n_labels=len(LABELS)).to(cfg.device)

bert_params = list(map(id, model.sent_encoder.parameters()))

new_params = filter(lambda p: id(p) not in bert_params, model.parameters())


optimizer = torch.optim.AdamW([
    {'params': model.sent_encoder.parameters(), 'lr': 2e-5}, 
    {'params': new_params, 'lr': 5e-5}                    
], weight_decay=cfg.weight_decay)


total_steps = math.ceil(len(train_loader) / cfg.accum_steps) * cfg.epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps
)


print(f"✅ 优化器已更新: 启用差分学习率 (BERT=2e-5, NewLayers=5e-5)")
print(f"✅ 模型参数 rule_alpha 初始值: {model.rule_alpha.item()}")



## Source training notebook cell 21


In [ ]:
from torch.amp import autocast

def train_one_epoch(loader):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    total_loss, n = 0.0, 0

    use_amp = cfg.device.startswith("cuda")
    use_bf16 = use_amp and torch.cuda.is_bf16_supported()
    amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
    device_type = "cuda" if use_amp else "cpu"

    scaler = torch.amp.GradScaler("cuda", enabled=(use_amp and not use_bf16))
    accum = cfg.accum_steps

    last_step = 0
    for step, batch in enumerate(loader, 1):
        last_step = step

        input_ids = batch["input_ids"].to(cfg.device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(cfg.device, non_blocking=True)
        feats = batch["feats"].to(cfg.device, non_blocking=True)
        sent_mask = batch["sent_mask"].to(cfg.device, non_blocking=True)
        y = batch["y"].to(cfg.device, non_blocking=True)

        with autocast(device_type=device_type, dtype=amp_dtype, enabled=use_amp):
            logits, _ = model(input_ids, attention_mask, feats, sent_mask)
            group_size = min(accum, len(loader) - ((step - 1) // accum) * accum)
            loss = criterion(logits, y) / group_size

        if scaler.is_enabled():
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if step % accum == 0:
            if scaler.is_enabled():
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)

            if scaler.is_enabled():
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            optimizer.zero_grad(set_to_none=True)
            scheduler.step()  

        total_loss += float(loss.item()) * group_size * y.size(0)
        n += y.size(0)


    if last_step != 0 and (last_step % accum) != 0:
        if scaler.is_enabled():
            scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)

        if scaler.is_enabled():
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()

        optimizer.zero_grad(set_to_none=True)
        scheduler.step()  

    return total_loss / max(n, 1)


## Source training notebook cell 22


In [ ]:
# H) 
def sigmoid_np(x):
    return 1 / (1 + np.exp(-x))

def f1_micro(y_true, y_pred):
    y_true = y_true.astype(int)
    y_pred = y_pred.astype(int)
    tp = (y_true & y_pred).sum()
    fp = ((1 - y_true) & y_pred).sum()
    fn = (y_true & (1 - y_pred)).sum()
    denom = (2 * tp + fp + fn)
    return float((2 * tp / denom) if denom > 0 else 0.0)

def f1_macro(y_true, y_pred):
    y_true = y_true.astype(int)
    y_pred = y_pred.astype(int)
    L = y_true.shape[1]
    f1s = []
    for i in range(L):
        yt = y_true[:, i]
        yp = y_pred[:, i]
        tp = (yt & yp).sum()
        fp = ((1 - yt) & yp).sum()
        fn = (yt & (1 - yp)).sum()
        denom = (2 * tp + fp + fn)
        f1s.append(float((2 * tp / denom) if denom > 0 else 0.0))
    return float(np.mean(f1s))

@torch.no_grad()
def eval_model(loader, thresholds=None, criterion=None):
    model.eval()

    all_logits = []
    all_y = []

    total_loss = 0.0
    total_n = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(cfg.device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(cfg.device, non_blocking=True)
        feats = batch["feats"].to(cfg.device, non_blocking=True)
        sent_mask = batch["sent_mask"].to(cfg.device, non_blocking=True)


        y = batch["y"].to(cfg.device, non_blocking=True).float()  

        logits, _ = model(input_ids, attention_mask, feats, sent_mask)  


        if criterion is not None:
            loss = criterion(logits, y)                  
            bs = y.size(0)
            total_loss += float(loss.item()) * bs        
            total_n += bs

        all_logits.append(logits.detach().cpu().numpy())
        all_y.append(y.detach().cpu().numpy())

    logits_np = np.concatenate(all_logits, axis=0)
    y_true = np.concatenate(all_y, axis=0).astype(int)
    
    FORGET_IDX = 1  
    
    TEMP_FORGET = globals().get("TEMP_FORGET", 1.0)  
    
    logits_adj = logits_np.copy()
    logits_adj[:, FORGET_IDX] = logits_adj[:, FORGET_IDX] / TEMP_FORGET
    
    probs = sigmoid_np(logits_adj)



    if thresholds is None:
        y_pred = (probs >= 0.5).astype(int)
    else:
        thr = np.array(thresholds, dtype=np.float32)[None, :]
        y_pred = (probs >= thr).astype(int)

    out = {
        "micro_f1": f1_micro(y_true, y_pred),
        "macro_f1": f1_macro(y_true, y_pred),
        "logits": logits_np,
        "probs": probs,
        "y_true": y_true,
    }

 
    if criterion is not None:
        out["loss"] = total_loss / max(total_n, 1)

    return out



## Source training notebook cell 23


In [ ]:
import os
import torch

best_micro = 0.0
best_path = os.path.join(cfg.out_dir, "best.pt")  # unified checkpoint name

patience = 4
wait = 0

print(f"Starting training (monitor: Micro-F1, patience: {patience})")

for epoch in range(cfg.epochs):
    tr_loss = train_one_epoch(train_loader)
    val_res = eval_model(val_loader, thresholds=None, criterion=criterion)

    val_loss = float(val_res["loss"])
    micro = float(val_res.get("micro_f1", 0.0))
    macro = float(val_res.get("macro_f1", 0.0))

    print(
        f"Epoch {epoch+1}/{cfg.epochs} | "
        f"train_loss={tr_loss:.4f} | val_loss={val_loss:.4f} | "
        f"val_microF1={micro:.4f} | val_macroF1={macro:.4f}"
    )

    if micro > best_micro:
        diff = micro - best_micro
        best_micro = micro
        wait = 0
        torch.save(model.state_dict(), best_path)
        print(f"  Saved new best checkpoint: {best_path} (Micro-F1={best_micro:.4f}, +{diff:.4f})")
    else:
        wait += 1
        print(f"  No improvement (wait={wait}/{patience})")
        if wait >= patience:
            print(f"Early stopping: no Micro-F1 improvement for {patience} epochs.")
            break

print("Training finished.")
print(f"Best validation Micro-F1: {best_micro:.4f}")
print(f"Best checkpoint path: {best_path}")


## Source training notebook cell 24


In [ ]:
import os
import json
import numpy as np
import torch

def _prf1(tp, fp, fn):
    p = tp / (tp + fp + 1e-9)
    r = tp / (tp + fn + 1e-9)
    f1 = 2 * p * r / (p + r + 1e-9)
    return float(p), float(r), float(f1)

# Ensure best checkpoint is loaded
best_path = os.path.join(cfg.out_dir, "best.pt")
assert os.path.exists(best_path), f"Missing checkpoint: {best_path}"
model.load_state_dict(torch.load(best_path, map_location=cfg.device))
model.eval()
print("Loaded best checkpoint for calibration:", best_path)

# Get validation probabilities
val_res = eval_model(val_loader, thresholds=None, criterion=None)
probs_val = val_res["probs"]          # (N, L)
y_val = val_res["y_true"].astype(int) # (N, L)

assert probs_val.ndim == 2 and y_val.ndim == 2, "Expected probs/y_true as (N, L)"
assert probs_val.shape == y_val.shape, "probs_val and y_val must have the same shape"
assert probs_val.shape[1] == len(LABELS), "LABELS length must match number of output labels"

print(f"Validation probs generated: N={probs_val.shape[0]}, L={probs_val.shape[1]}")

# Threshold search settings
grid = np.arange(0.05, 0.951, 0.01, dtype=np.float32)

# Constraint for suicide intent
SUICIDE_LABEL = "suicide intent"
MIN_PRECISION = 0.70
TARGET_RECALL = 0.85

thresholds = []

print("\nPer-label threshold optimization on validation set:")
print(f"{'Label':<28} | {'Thr':>5} | {'Prec':>6} | {'Rec':>6} | {'F1':>6} | {'Rule':<12}")
print("-" * 80)

for j, lab in enumerate(LABELS):
    yt = y_val[:, j]
    pv = probs_val[:, j]

    if yt.sum() == 0:
        thr = 0.50
        thresholds.append(thr)
        print(f"{lab:<28} | {thr:5.2f} | {0.0:6.3f} | {0.0:6.3f} | {0.0:6.3f} | No positives")
        continue

    best_thr = 0.50
    best_p, best_r, best_f1 = 0.0, 0.0, -1.0
    rule_note = "MaxF1"

    if lab == SUICIDE_LABEL:
        best_r_under = -1.0
        best_f1_under = -1.0
        best_thr_under = 0.50
        best_p_uc = best_r_uc = best_f1_uc = 0.0

        for thr in grid:
            yp = (pv >= thr).astype(int)
            tp = int(((yt == 1) & (yp == 1)).sum())
            fp = int(((yt == 0) & (yp == 1)).sum())
            fn = int(((yt == 1) & (yp == 0)).sum())
            p, r, f1 = _prf1(tp, fp, fn)

            if p >= MIN_PRECISION:
                if (r > best_r_under) or (r == best_r_under and f1 > best_f1_under):
                    best_r_under = r
                    best_f1_under = f1
                    best_thr_under = float(thr)
                    best_p_uc, best_r_uc, best_f1_uc = p, r, f1

        if best_r_under >= 0:
            best_thr, best_p, best_r, best_f1 = best_thr_under, best_p_uc, best_r_uc, best_f1_uc
            rule_note = f"Prec≥{MIN_PRECISION:.2f}"
            if best_r < TARGET_RECALL:
                rule_note += " (best-effort)"
        else:
            for thr in grid:
                yp = (pv >= thr).astype(int)
                tp = int(((yt == 1) & (yp == 1)).sum())
                fp = int(((yt == 0) & (yp == 1)).sum())
                fn = int(((yt == 1) & (yp == 0)).sum())
                p, r, f1 = _prf1(tp, fp, fn)
                if f1 > best_f1:
                    best_thr, best_p, best_r, best_f1 = float(thr), p, r, f1
            rule_note = "FallbackMaxF1"
    else:
        for thr in grid:
            yp = (pv >= thr).astype(int)
            tp = int(((yt == 1) & (yp == 1)).sum())
            fp = int(((yt == 0) & (yp == 1)).sum())
            fn = int(((yt == 1) & (yp == 0)).sum())
            p, r, f1 = _prf1(tp, fp, fn)
            if f1 > best_f1:
                best_thr, best_p, best_r, best_f1 = float(thr), p, r, f1
        rule_note = "MaxF1"

    thresholds.append(best_thr)
    print(f"{lab:<28} | {best_thr:5.2f} | {best_p:6.3f} | {best_r:6.3f} | {best_f1:6.3f} | {rule_note:<12}")

# Save thresholds.json
thr_path = os.path.join(cfg.out_dir, "thresholds.json")
with open(thr_path, "w", encoding="utf-8") as f:
    json.dump({"labels": LABELS, "thresholds": thresholds}, f, ensure_ascii=False, indent=2)

print("-" * 80)
print("Thresholds saved to:", thr_path)

# Final test evaluation with optimized thresholds
test_res = eval_model(test_loader, thresholds=thresholds, criterion=None)
print("\nFinal test evaluation (optimized thresholds):")
print(f"  Micro-F1: {test_res['micro_f1']:.4f}")
print(f"  Macro-F1: {test_res['macro_f1']:.4f}")


## Source training notebook cell 25


In [ ]:
@torch.no_grad()
def predict_with_rationales(text: str, topk=3):
    model.eval()
    sents = sent_split(str(text))
    if len(sents) == 0:
        sents = [""]

    if len(sents) > cfg.max_sents:
        positions = np.unique(np.round(np.linspace(0, len(sents)-1, cfg.max_sents)).astype(int))
        sents = [sents[i] for i in positions]
    feats = np.stack([context_feats(s) for s in sents], axis=0)

    enc = tokenizer(
        sents,
        padding=True,
        truncation=True,
        max_length=cfg.max_sent_len,
        return_tensors="pt",
    )

    sent_mask = torch.tensor([[1 if s.strip() else 0 for s in sents]], dtype=torch.bool)
    feats_3d = torch.tensor(feats[None, :, :], dtype=torch.float32)

    logits, attn = model(
        enc["input_ids"].to(cfg.device),
        enc["attention_mask"].to(cfg.device),
        feats_3d.to(cfg.device),
        sent_mask.to(cfg.device),
    )

    probs = torch.sigmoid(logits).squeeze(0).cpu().numpy()
    attn = attn.squeeze(0).cpu().numpy()  # (L,S)

    pred = (probs >= np.array(thresholds)).astype(int)

    results = []
    S = len(sents)
    for j, lab in enumerate(LABELS):
        weights = attn[j]
        idxs = np.argsort(-weights)[:min(topk, S)]
        rationales = [(int(i), float(weights[i]), sents[i]) for i in idxs if sents[i].strip()]
        results.append({"label": lab, "prob": float(probs[j]), "pred": int(pred[j]), "top_sentences": rationales})
    return results



demo_text = test_df.iloc[0]["text_clean"]
res = predict_with_rationales(demo_text, topk=3)

for r in res:
    if r["pred"] == 1:
        print(f"\n[{r['label']}] prob={r['prob']:.3f}")
        for i, w, s in r["top_sentences"]:
            print(f"  - (sent#{i}, w={w:.3f}) {s[:160]}...")

## Source training notebook cell 26


In [ ]:
import os
import json
import numpy as np
import pandas as pd

# -----------------------------
# Load thresholds from JSON
# -----------------------------
thr_path = os.path.join(cfg.out_dir, "thresholds.json")
assert os.path.exists(thr_path), f"Missing thresholds file: {thr_path}"

with open(thr_path, "r", encoding="utf-8") as f:
    thr_data = json.load(f)

LABELS_FROM_JSON = thr_data["labels"]
thr_list = thr_data["thresholds"]
thr_map = dict(zip(LABELS_FROM_JSON, thr_list))

print("Loaded thresholds from:", thr_path)
print("Number of labels:", len(LABELS_FROM_JSON))

# -----------------------------
# Get test probabilities (only if not already available)
# -----------------------------
if "test_res" not in locals() or "probs" not in test_res:
    test_res = eval_model(test_loader, thresholds=None, criterion=None)

probs_test = test_res["probs"]
assert probs_test.ndim == 2, "probs_test must be (N, L)"
assert probs_test.shape[1] == len(LABELS_FROM_JSON), "Label count must match probs columns"

# -----------------------------
# Risk assignment (core=3)
# -----------------------------
def assign_risk_level(row_probs, labels, thr_map):
    p = dict(zip(labels, row_probs))

    # Red: suicide intent
    if p.get("suicide intent", 0.0) >= thr_map.get("suicide intent", 0.5):
        return "Red"

    # Amber: core emotions (core=3)
    core = ["hopelessness", "worthlessness", "sadness"]
    for lab in core:
        if lab not in labels:
            raise ValueError(f"Missing label in labels list: {lab}")
        if lab not in thr_map:
            raise ValueError(f"Missing threshold in thr_map: {lab}")

    for lab in core:
        if p.get(lab, 0.0) >= thr_map.get(lab, 0.5):
            return "Amber"

    return "Green"

risk_levels = [assign_risk_level(row, LABELS_FROM_JSON, thr_map) for row in probs_test]

# -----------------------------
# Summarize risk distribution
# -----------------------------
df_risk = pd.DataFrame({"risk": risk_levels})
counts = df_risk["risk"].value_counts()
props = df_risk["risk"].value_counts(normalize=True)

summary = pd.DataFrame({
    "count": counts,
    "proportion": props,
    "percent": (props * 100).round(2)
}).reindex(["Green", "Amber", "Red"])

print("\nRisk distribution (count, proportion, percent):")
print(summary)


## Source training notebook cell 27


In [ ]:

import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, hamming_loss


y_true_test = test_res["y_true"]
probs_test = test_res["probs"]


y_pred_test = np.zeros_like(probs_test)
for i, thr in enumerate(thresholds):
    y_pred_test[:, i] = (probs_test[:, i] >= thr).astype(int)


print("正在生成性能报告...")
report_dict = classification_report(
    y_true_test, 
    y_pred_test, 
    target_names=LABELS, 
    output_dict=True, 
    zero_division=0
)

df_report = pd.DataFrame(report_dict).transpose()


h_loss = hamming_loss(y_true_test, y_pred_test)

exact_match = accuracy_score(y_true_test, y_pred_test)

print(f"Hamming Loss: {h_loss:.4f}")
print(f"Exact Match Ratio: {exact_match:.4f}")


output_csv_path = os.path.join(cfg.out_dir, "final_test_performance_metrics.csv")
output_xlsx_path = os.path.join(cfg.out_dir, "final_test_performance_metrics.xlsx")


df_report.to_csv(output_csv_path)
print(f"\n✅ 详细性能报告已保存为 CSV: {output_csv_path}")


try:
    df_report.to_excel(output_xlsx_path)
    print(f"✅ 详细性能报告已保存为 Excel: {output_xlsx_path}")
except ImportError:
    print("⚠️ 未安装 openpyxl，跳过 Excel 保存 (CSV 已保存)。如需 Excel 请运行: pip install openpyxl")


print("\n=== 性能报告预览 ===")
display(df_report)

## Source training notebook cell 33


In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, hamming_loss


def get_probs_and_true(res_dict):
    assert isinstance(res_dict, dict), "eval_model must return a dict"
    assert "probs" in res_dict, f"Missing 'probs' in eval_model output. keys={list(res_dict.keys())}"

    probs = res_dict["probs"]

    y_true = None
    for k in ["y_true", "targets", "labels", "y", "true", "gold"]:
        if k in res_dict and res_dict[k] is not None:
            y_true = res_dict[k]
            break

    assert y_true is not None, f"Missing ground-truth labels in eval_model output. keys={list(res_dict.keys())}"

    probs = np.asarray(probs)
    y_true = np.asarray(y_true).astype(int)

    assert probs.ndim == 2 and y_true.ndim == 2, f"Expected 2D arrays; got probs{probs.shape}, y_true{y_true.shape}"
    assert probs.shape == y_true.shape, f"Shape mismatch: probs{probs.shape} vs y_true{y_true.shape}"
    return probs, y_true


def build_thresholds_list(labels, thresholds):
    """
    thresholds can be:
      - float (e.g., 0.5)
      - list/np.ndarray of len L (aligned with labels)
      - dict mapping label -> threshold
    Returns: list of len L aligned with labels
    """
    L = len(labels)
    if isinstance(thresholds, (float, int)):
        return [float(thresholds)] * L

    if isinstance(thresholds, dict):
        missing = [lab for lab in labels if lab not in thresholds]
        assert not missing, f"thresholds dict missing labels: {missing}"
        return [float(thresholds[lab]) for lab in labels]

   
    thr_arr = list(thresholds)
    assert len(thr_arr) == L, f"threshold list length {len(thr_arr)} != number of labels {L}"
    return [float(x) for x in thr_arr]


def probs_to_preds(probs, thr_list):
    thr = np.asarray(thr_list)[None, :]  
    return (probs >= thr).astype(int)


thr_default_list = build_thresholds_list(LABELS, 0.5)
res_default = eval_model(test_loader, thresholds=thr_default_list)

thr_cal_list = build_thresholds_list(LABELS, thresholds)
res_cal = eval_model(test_loader, thresholds=thr_cal_list)


probs_def, y_true = get_probs_and_true(res_default)
probs_cal, y_true2 = get_probs_and_true(res_cal)
assert np.array_equal(y_true, y_true2), "y_true differs between runs; check loader shuffle / eval pipeline."

pred_def = probs_to_preds(probs_def, thr_default_list)
pred_cal = probs_to_preds(probs_cal, thr_cal_list)


fp_def_total = int(((pred_def == 1) & (y_true == 0)).sum())
fp_cal_total = int(((pred_cal == 1) & (y_true == 0)).sum())

fp_def_per_label = ((pred_def == 1) & (y_true == 0)).sum(axis=0)
fp_cal_per_label = ((pred_cal == 1) & (y_true == 0)).sum(axis=0)


prec_def_micro = precision_score(y_true, pred_def, average="micro", zero_division=0)
prec_def_macro = precision_score(y_true, pred_def, average="macro", zero_division=0)

prec_cal_micro = precision_score(y_true, pred_cal, average="micro", zero_division=0)
prec_cal_macro = precision_score(y_true, pred_cal, average="macro", zero_division=0)


hl_def = hamming_loss(y_true, pred_def)
hl_cal = hamming_loss(y_true, pred_cal)


summary = pd.DataFrame({
    "Setting": ["Default thr=0.5", "Calibrated thr"],
    "FP_total": [fp_def_total, fp_cal_total],
    "Precision_micro": [prec_def_micro, prec_cal_micro],
    "Precision_macro": [prec_def_macro, prec_cal_macro],
    "HammingLoss": [hl_def, hl_cal],
})

print("\n=== RO3/RQ3 Evidence: Before vs After Calibration (Test Set) ===")
print(summary.to_string(index=False))


fp_table = pd.DataFrame({
    "label": LABELS,
    "FP_default_0.5": fp_def_per_label.astype(int),
    "FP_calibrated": fp_cal_per_label.astype(int),
    "FP_reduction": (fp_def_per_label - fp_cal_per_label).astype(int),
}).sort_values("FP_reduction", ascending=False)

print("\n=== Per-label FP change (Top 15 by FP reduction) ===")
print(fp_table.head(15).to_string(index=False))



## Source training notebook cell 41


In [ ]:
import numpy as np
import pandas as pd

# Deltas you want in the table
deltas = [-0.05, -0.02, 0.02, 0.05]

# base threshold map must be a dict: {label: threshold}
base_thr_map = dict(thr_map)  # make a safe copy

N = probs_test.shape[0]

def get_risk_list(thr_map_use):
    return [assign_risk_level(probs_test[i], LABELS_FROM_JSON, thr_map_use) for i in range(N)]

base_risk = get_risk_list(base_thr_map)

rows = []
for d in deltas:
    new_thr_map = {
        lab: float(np.clip(base_thr_map[lab] + d, 0.0, 1.0))
        for lab in base_thr_map
    }
    new_risk = get_risk_list(new_thr_map)

    base_s = pd.Series(base_risk, name="base")
    new_s  = pd.Series(new_risk,  name="new")

    rows.append({
        "Δ Threshold": f"{d:+.2f}",
        "Unchanged (%)": round((base_s == new_s).mean() * 100, 2),
        "Green→Amber (%)": round(((base_s=="Green") & (new_s=="Amber")).mean() * 100, 2),
        "Amber→Red (%)": round(((base_s=="Amber") & (new_s=="Red")).mean() * 100, 2),
        "Amber→Green (%)": round(((base_s=="Amber") & (new_s=="Green")).mean() * 100, 2),
        "Red→Amber (%)": round(((base_s=="Red") & (new_s=="Amber")).mean() * 100, 2),
    })

df_stability = pd.DataFrame(rows)
df_stability
